In [1]:
import numpy as np
import pandas as pd
from aequilibrae.paths.public_transport import HyperpathGenerating
from numba import jit

RS = 124  # random seed

In [2]:
def create_vertices(n):
    x = np.linspace(0, 1, n)
    y = np.linspace(0, 1, n)
    xv, yv = np.meshgrid(x, y, indexing="xy")
    vertices = pd.DataFrame()
    vertices["x"] = xv.ravel()
    vertices["y"] = yv.ravel()
    return vertices

n = 3
vertices = create_vertices(n)
vertices

,x,y
0,0.0,0.0
1,0.5,0.0
2,1.0,0.0
3,0.0,0.5
4,0.5,0.5
5,1.0,0.5
6,0.0,1.0
7,0.5,1.0
8,1.0,1.0


In [3]:
@jit
def create_edges_numba(n):
    m = 2 * n * (n - 1)
    tail = np.zeros(m, dtype=np.uint32)
    head = np.zeros(m, dtype=np.uint32)
    k = 0
    for i in range(n - 1):
        for j in range(n):
            tail[k] = i + j * n
            head[k] = i + 1 + j * n
            k += 1
            tail[k] = j + i * n
            head[k] = j + (i + 1) * n
            k += 1
    return tail, head

def create_edges(n, seed=124):
    tail, head = create_edges_numba(n)
    edges = pd.DataFrame()
    edges["tail"] = tail
    edges["head"] = head
    m = len(edges)
    rng = np.random.default_rng(seed=seed)
    edges["trav_time"] = rng.uniform(0.0, 1.0, m)
    edges["delay_base"] = rng.uniform(0.0, 1.0, m)
    edges['var_1_c'] = rng.uniform(0.0, 1.0, m)
    return edges

In [4]:
edges = create_edges(n, seed=RS)

In [5]:
alpha = 10.0

delay_base = edges.delay_base.values
indices = np.where(delay_base == 0.0)
delay_base[indices] = 1.0 # use this to prevent an error?
freq_base = 1.0 / delay_base
freq_base[indices] = np.inf
edges["freq_base"] = freq_base

if alpha == 0.0:
    edges["freq"] = np.inf
else:
    edges["freq"] = edges.freq_base / alpha

In [6]:
edges['demand'] = 1

centroids = np.array([0,3,8])

# # Spiess & Florian
sf = HyperpathGenerating(
    edges, tail="tail", head="head", trav_time="trav_time", freq="freq", 
    skim_cols = ["trav_time"], centroids = centroids
)

# Spiess & Florian
# sf = HyperpathGenerating(
#     edges, tail="tail", head="head", trav_time="trav_time", freq="freq"
# )

In [7]:
origin_column = np.array([0, 3])
destination_column = np.array([8, 8])
demand_column = np.array([1, 1])
sf.assign(origin_column, destination_column, demand_column)

In [8]:
sf.skim_u_i

,0,3,8
0,0.000000e+00,3.356967e+00,15.013191
3,1.797693e+308,0.000000e+00,12.450906
8,1.797693e+308,1.797693e+308,0.000000


In [9]:
sf._edges

,freq,tail,trav_time,head,edge_idx,volume
0,0.101065,0,0.785253,1,0,0.206254
1,0.388937,0,0.785859,3,1,0.793746
2,0.139711,3,0.969136,4,2,0.138940
3,0.197673,1,0.748060,4,3,0.019252
4,0.150577,6,0.655551,7,4,1.654807
5,0.142381,2,0.938885,5,5,0.187002
6,1.920114,1,0.178614,2,6,0.187002
7,1.663991,3,0.588647,6,7,1.654807
8,0.105781,4,0.442799,5,8,0.033121
9,0.399440,4,0.348847,7,9,0.125070
